# 35. Reasoning — 단계적 사고

> **제35장** · **이론편 대응: 23.6절 (Reasoning)**
> **예상 소요**: 70분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음
> **API 키**: 선택 (없으면 개념·수치 실험으로 대체)

---

## 이 장에서 하는 일

**"단계적으로 생각해 보자"는 한 문장이 성능을 바꾼다.** 왜 그럴까.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 왜 계산을 틀리는가 | 23.6절 |
| 2 | **Chain-of-Thought** | 23.6절 |
| 3 | **왜 작동하는가 — 확률로 설명** ★ | 20.1절, 23.6절 |
| 4 | Zero-shot vs Few-shot CoT | 23.6절 |
| 5 | **Self-Consistency 검증** ★ | 23.6절 |
| 6 | 추론 특화 모델 | 23.6절 |
| 7 | 한계와 주의점 | 23.6절 |

**3절과 5절이 핵심이다.** "왜 되는가"를 확률 계산으로 설명하고,
다수결이 정확도를 얼마나 올리는지 직접 계산한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import os
from pathlib import Path
from math import comb

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)

# API 키 확인 (25장과 같은 방식)
root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
try:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
except ImportError:
    pass

API_KEY, BASE_URL, MODEL = None, None, "gpt-4o-mini"
for env_name, base, model in [
        ("OPENAI_API_KEY", None, "gpt-4o-mini"),
        ("GROQ_API_KEY", "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
        ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash")]:
    if os.getenv(env_name):
        API_KEY, BASE_URL, MODEL = os.getenv(env_name), base, model
        print(f"API 키 발견: {env_name}  (모델: {MODEL})")
        break

if not API_KEY:
    print("API 키 없음 — 3절과 5절의 수치 실험은 그대로 실행됩니다.")
    print("  실제 모델 호출이 필요한 부분은 안내로 대체합니다.")

---

## 1. 왜 계산을 틀리는가 — 이론편 23.6절

23장에서 봤듯 LLM은 **다음 토큰을 예측**할 뿐이다. 계산기를 내장하고 있지 않다.

그런데도 간단한 산수는 곧잘 맞힌다. **학습 데이터에 그런 계산이 많이 있었기 때문**이다.
문제는 학습에서 본 적 없는 조합이 나올 때다.

$$\text{"23 × 17 = "} \to ?$$

모델은 이 계산을 하는 것이 아니라, **"이 다음에 올 법한 숫자"**를 고른다.

In [ ]:
def ask_llm(messages, max_tokens=400, temperature=0.0, n=1):
    """LLM 호출 — 키가 없으면 None"""
    if not API_KEY:
        return None
    try:
        from openai import OpenAI
        kwargs = {"api_key": API_KEY}
        if BASE_URL:
            kwargs["base_url"] = BASE_URL
        client = OpenAI(**kwargs)
        r = client.chat.completions.create(
            model=MODEL, messages=messages,
            max_tokens=max_tokens, temperature=temperature, n=n)
        if n == 1:
            return r.choices[0].message.content
        return [c.message.content for c in r.choices]
    except Exception as e:
        print(f"[오류] {type(e).__name__}: {str(e)[:120]}")
        return None


# 여러 단계를 거쳐야 하는 문제들
problems = [
    {
        "question": "한 상자에 사과가 12개 들어 있습니다. 상자 7개를 사서 "
                    "친구 5명에게 똑같이 나눠 주면 한 명당 몇 개인가요?",
        "answer": 16.8,
        "steps": "12 x 7 = 84,  84 / 5 = 16.8",
    },
    {
        "question": "책이 한 권에 13,500원입니다. 4권을 사고 10,000원 할인 쿠폰을 "
                    "쓰면 얼마를 내나요?",
        "answer": 44000,
        "steps": "13500 x 4 = 54000,  54000 - 10000 = 44000",
    },
    {
        "question": "시속 60km로 2시간 30분을 달린 뒤, 시속 80km로 1시간 15분을 "
                    "더 달렸습니다. 총 몇 km인가요?",
        "answer": 250,
        "steps": "60 x 2.5 = 150,  80 x 1.25 = 100,  150 + 100 = 250",
    },
]

print("=" * 70)
print("여러 단계가 필요한 문제")
print("=" * 70)
for i, p in enumerate(problems, 1):
    print(f"\n[{i}] {p['question']}")
    print(f"    풀이: {p['steps']}")
    print(f"    정답: {p['answer']}")

print()
print("-" * 70)
print("각 문제가 두세 번의 계산을 거쳐야 한다.")
print("한 번에 답을 내려면 그 모든 계산을 '한 토큰 안에' 담아야 한다.")

In [ ]:
print("=" * 70)
print("바로 답하게 하면")
print("=" * 70)

for i, p in enumerate(problems, 1):
    msgs = [{"role": "user",
             "content": f"{p['question']}\n\n숫자만 답하세요."}]
    answer = ask_llm(msgs, max_tokens=20)

    print(f"\n[{i}] {p['question'][:44]}...")
    if answer:
        print(f"    모델 답: {answer.strip()}")
        print(f"    정답   : {p['answer']}")
    else:
        print(f"    (API 키 없음 — 정답은 {p['answer']})")

if not API_KEY:
    print()
    print("-" * 70)
    print("이 실험의 요지")
    print("  '숫자만 답하라'고 하면 모델은 중간 계산을 표현할 자리가 없다.")
    print("  그 모든 과정을 한 번의 토큰 예측으로 압축해야 한다.")
    print("  단계가 늘어날수록 틀릴 확률이 커진다.")

---

## 2. Chain-of-Thought — 이론편 23.6절

이론편 23.6절에서 다룬 해법은 놀랍도록 단순하다.

> **"단계적으로 생각해 보자(Let's think step by step)"**

이 한 문장을 덧붙이면 모델이 중간 과정을 적어 나가고, 정답률이 오른다.

In [ ]:
print("=" * 70)
print("CoT 적용")
print("=" * 70)

for i, p in enumerate(problems, 1):
    msgs = [{"role": "user",
             "content": f"{p['question']}\n\n단계적으로 생각해서 풀어 주세요."}]
    answer = ask_llm(msgs, max_tokens=300)

    print(f"\n[{i}] {p['question'][:44]}...")
    if answer:
        for line in answer.strip().split("\n")[:8]:
            print(f"    {line}")
        print(f"    (정답: {p['answer']})")
    else:
        print(f"    (API 키 없음)")
        print(f"    기대하는 풀이: {p['steps']}")

print()
print("=" * 70)
print("무엇이 달라졌나")
print("  모델이 중간 계산을 텍스트로 적는다.")
print("  '12 x 7 = 84' 를 쓰고 나면, 다음 단계는 84 를 보고 계산하면 된다.")
print()
print("  즉 어려운 문제 하나를 **쉬운 문제 여러 개로 쪼갠 것**이다.")
print("  19장의 Diffusion 과 같은 발상이다.")

---

## 3. 왜 작동하는가 — 확률로 설명 ★

**"생각하는 척"이 아니라 실제로 도움이 된다.** 이유를 세 가지로 나눠 보자.

### 이유 1 — 계산 자원이 늘어난다

23장에서 봤듯 **토큰 하나를 만들 때 신경망을 한 번 통과**한다.
답을 바로 내면 계산 기회가 한 번뿐이지만, 100토큰을 쓰면 100번 계산할 수 있다.

$$\text{답만 출력} \to \text{신경망 1회} \qquad \text{CoT 100토큰} \to \text{신경망 100회}$$

### 이유 2 — 중간 결과가 문맥에 남는다

한 번 "84"를 적어 두면, 그 다음 계산은 **문맥에서 84를 읽어** 진행한다.
머릿속에 담아 둘 필요가 없다.

### 이유 3 — 각 단계가 쉬운 문제가 된다

이것이 가장 중요하다. **확률로 계산해 보자.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("단계 수와 정답률 (이론편 23.6절)")
print("=" * 70)
print()
print("가정: 각 단계를 맞힐 확률이 p 일 때, n단계를 모두 맞힐 확률 = p^n")
print()
print(f"{'단계 정확도 p':<16}{'3단계':<12}{'5단계':<12}{'10단계'}")
print("-" * 70)
for p in [0.7, 0.8, 0.9, 0.95, 0.99]:
    print(f"{p:<16}{p**3:<12.3f}{p**5:<12.3f}{p**10:.3f}")
print("-" * 70)
print()
print("이 표를 두 방향으로 읽을 수 있다.")
print()
print("[나쁜 소식] 단계가 늘면 정답률이 급격히 떨어진다")
print(f"  p=0.9 라도 10단계면 {0.9**10:.1%} 밖에 안 된다.")
print()
print("[좋은 소식] 각 단계를 쉽게 만들면 p 가 올라간다")
print(f"  p 를 0.7 → 0.95 로 올리면, 5단계 정답률이 {0.7**5:.1%} → {0.95**5:.1%}")
print()
print("CoT가 하는 일이 바로 이것이다.")
print("  '전체를 한 번에' 대신 '쉬운 단계 여러 개'로 바꿔 p 를 높인다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: 단계 수에 따른 정답률 ---
ax = axes[0]
steps = np.arange(1, 16)
for p, color in [(0.7, "#DC2626"), (0.8, "#EA580C"),
                 (0.9, "#0D9488"), (0.95, "#1E40AF")]:
    ax.plot(steps, p ** steps, marker="o", markersize=3,
            linewidth=2, color=color, label=f"p={p}")
ax.set_xlabel("단계 수")
ax.set_ylabel("전체 정답률")
ax.set_title("단계가 늘어날수록")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- 오른쪽: 같은 문제를 어떻게 나누느냐 ---
ax = axes[1]
scenarios = {
    "한 번에 (어려움)\np=0.5, 1단계": 0.5 ** 1,
    "3단계로 쪼갬\np=0.85, 3단계": 0.85 ** 3,
    "5단계로 쪼갬\np=0.95, 5단계": 0.95 ** 5,
}
bars = ax.bar(range(3), list(scenarios.values()),
              color=["#DC2626", "#EA580C", "#0D9488"])
for b, v in zip(bars, scenarios.values()):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f"{v:.3f}",
            ha="center", fontsize=10)
ax.set_xticks(range(3))
ax.set_xticklabels(list(scenarios.keys()), fontsize=8)
ax.set_ylabel("정답률")
ax.set_title("쪼개면 각 단계가 쉬워진다")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("오른쪽 그래프가 CoT의 요지다.")
print("  단계를 늘리면 곱하는 횟수는 늘지만, 각 단계의 p 가 훨씬 커진다.")
print("  결과적으로 전체 정답률이 올라간다.")

In [ ]:
import numpy as np

print("=" * 70)
print("계산 자원 관점 (이유 1)")
print("=" * 70)
print()
print("23장 5절에서 봤듯 토큰 하나마다 신경망을 한 번 통과한다.")
print()
print(f"{'방식':<24}{'출력 토큰':<14}{'신경망 통과 횟수'}")
print("-" * 70)
print(f"{'답만 출력 (예: 4000)':<24}{'약 3':<14}{'3회'}")
print(f"{'짧은 CoT':<24}{'약 50':<14}{'50회'}")
print(f"{'자세한 CoT':<24}{'약 200':<14}{'200회'}")
print("-" * 70)
print()
print("계산량이 수십 배 늘어난다.")
print("  대신 시간과 비용도 그만큼 든다 (25장 6절).")
print()
print("이것이 '추론 모델'이 비싸고 느린 이유이기도 하다 (6절).")

---

## 4. Zero-shot vs Few-shot CoT — 이론편 23.6절

CoT를 유도하는 방법이 두 가지다.

| 방식 | 하는 일 | 장단점 |
|---|---|---|
| **Zero-shot CoT** | "단계적으로 생각하자"만 덧붙임 | 간단, 예시 불필요 |
| **Few-shot CoT** | 풀이 과정이 담긴 예시를 보여줌 | 형식을 정확히 제어 가능 |

In [ ]:
print("=" * 70)
print("Zero-shot CoT")
print("=" * 70)
print()
print("프롬프트 형태")
print("""
  [질문]
  ...

  단계적으로 생각해 봅시다.
""")
print()
print("=" * 70)
print("Few-shot CoT")
print("=" * 70)
print()
print("프롬프트 형태 — 풀이 예시를 먼저 보여준다")
print("""
  Q: 연필 5자루가 1500원입니다. 12자루는 얼마인가요?
  A: 한 자루 가격을 구합니다. 1500 / 5 = 300원.
     12자루면 300 x 12 = 3600원.
     답: 3600원

  Q: [실제 질문]
  A:
""")
print()
print("-" * 70)
print("Few-shot 의 장점")
print("  - 풀이 형식을 원하는 대로 맞출 수 있다")
print("  - 답변 형태가 일정해 파싱하기 쉽다 (25장 7절)")
print()
print("Few-shot 의 단점")
print("  - 예시가 프롬프트를 차지해 비용이 는다")
print("  - 좋은 예시를 만드는 데 품이 든다")

In [ ]:
print("=" * 70)
print("세 방식 비교 실행")
print("=" * 70)

problem = problems[2]      # 속도 문제

few_shot_example = """Q: 연필 5자루가 1500원입니다. 12자루는 얼마인가요?
A: 한 자루 가격을 구합니다. 1500 / 5 = 300원.
   12자루면 300 x 12 = 3600원.
   답: 3600원

Q: """

methods = {
    "바로 답하기": [
        {"role": "user", "content": f"{problem['question']}\n\n숫자만 답하세요."}
    ],
    "Zero-shot CoT": [
        {"role": "user", "content": f"{problem['question']}\n\n단계적으로 생각해 봅시다."}
    ],
    "Few-shot CoT": [
        {"role": "user", "content": few_shot_example + problem["question"] + "\nA:"}
    ],
}

print(f"문제: {problem['question']}")
print(f"정답: {problem['answer']} km")
print()

for name, msgs in methods.items():
    print(f"\n[{name}]")
    ans = ask_llm(msgs, max_tokens=250)
    if ans:
        for line in ans.strip().split("\n")[:6]:
            print(f"  {line}")
    else:
        print("  (API 키 없음)")

if not API_KEY:
    print()
    print("-" * 70)
    print("기대하는 차이")
    print("  바로 답하기  : 중간 과정 없이 숫자 하나")
    print("  Zero-shot   : 자유로운 형식으로 단계 서술")
    print("  Few-shot    : 예시와 같은 형식으로 서술")

---

## 5. Self-Consistency ★ — 이론편 23.6절

CoT를 여러 번 실행하면 **매번 다른 풀이 경로**가 나온다(24장 4절의 샘플링).

**여러 번 풀어 보고 다수결로 정하면** 정확도가 오른다. 이것이 Self-Consistency다.

**얼마나 오를까? 확률로 계산해 보자.**

In [ ]:
from math import comb
import numpy as np


def majority_correct_prob(p, k):
    """개별 정확도 p로 k번 시도했을 때, 과반이 정답일 확률

    이항분포로 계산한다 (이론편 6.3절).
    k번 중 i번 맞을 확률 = C(k,i) * p^i * (1-p)^(k-i)
    이것을 i가 과반인 경우에 대해 모두 더한다.
    """
    threshold = k // 2 + 1
    return sum(comb(k, i) * p**i * (1-p)**(k-i) for i in range(threshold, k+1))


print("=" * 70)
print("Self-Consistency 효과 (이론편 23.6절)")
print("=" * 70)
print()
print(f"{'개별 정확도':<14}{'1회':<12}{'3회':<12}{'5회':<12}{'9회'}")
print("-" * 70)
for p in [0.5, 0.6, 0.7, 0.8, 0.9]:
    row = f"{p:<14}{p:<12.3f}"
    for k in [3, 5, 9]:
        row += f"{majority_correct_prob(p, k):<12.3f}"
    print(row)
print("-" * 70)
print()
print("주목할 점")
print()
print("[1] p > 0.5 이면 시도를 늘릴수록 정확해진다")
print(f"    p=0.7: 1회 0.700 → 9회 {majority_correct_prob(0.7, 9):.3f}")
print()
print("[2] p = 0.5 면 아무리 늘려도 0.5 다")
print(f"    p=0.5: 9회에도 {majority_correct_prob(0.5, 9):.3f}")
print("    동전 던지기를 여러 번 해도 앞면 확률은 그대로다.")
print()
print("[3] p < 0.5 면 오히려 나빠진다")
print(f"    p=0.4: 1회 0.400 → 9회 {majority_correct_prob(0.4, 9):.3f}")
print()
print("→ 모델이 '절반보다는 낫게' 풀 수 있어야 효과가 있다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- 왼쪽: 시도 횟수에 따른 향상 ---
ax = axes[0]
ks = [1, 3, 5, 7, 9, 11, 15, 21]
for p, color in [(0.4, "#DC2626"), (0.5, "#94A3B8"),
                 (0.6, "#EA580C"), (0.7, "#0D9488"), (0.8, "#1E40AF")]:
    probs = [majority_correct_prob(p, k) if k > 1 else p for k in ks]
    ax.plot(ks, probs, marker="o", markersize=4, linewidth=2,
            color=color, label=f"p={p}")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("시도 횟수 (k)")
ax.set_ylabel("다수결 정답률")
ax.set_title("Self-Consistency 효과")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- 오른쪽: p=0.7에서 향상폭 ---
ax = axes[1]
p = 0.7
ks2 = [1, 3, 5, 9, 15]
probs2 = [majority_correct_prob(p, k) if k > 1 else p for k in ks2]
bars = ax.bar(range(len(ks2)), probs2, color="#0D9488")
for b, v, k in zip(bars, probs2, ks2):
    ax.text(b.get_x()+b.get_width()/2, v+0.015, f"{v:.3f}",
            ha="center", fontsize=9)
ax.axhline(p, color="#DC2626", linestyle="--", linewidth=1.5)
ax.text(3.5, p-0.04, "1회 시도", fontsize=8, color="#DC2626")
ax.set_xticks(range(len(ks2)))
ax.set_xticklabels([f"{k}회" for k in ks2])
ax.set_ylabel("정답률")
ax.set_title(f"개별 정확도 {p} 일 때")
ax.set_ylim(0.6, 1.0)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("향상폭이 점점 줄어든다 (수확 체감).")
print(f"  1회 → 3회: +{majority_correct_prob(0.7,3)-0.7:.3f}")
print(f"  9회 → 15회: +{majority_correct_prob(0.7,15)-majority_correct_prob(0.7,9):.3f}")
print()
print("비용은 시도 횟수에 비례해 늘어나므로, 5~10회 정도가 실용적이다.")

In [ ]:
import re
from collections import Counter

print("=" * 70)
print("Self-Consistency 구현")
print("=" * 70)


def extract_number(text):
    """응답에서 최종 숫자를 뽑아낸다"""
    if not text:
        return None
    # '답:' 뒤의 숫자를 우선 찾고, 없으면 마지막 숫자
    m = re.search(r"답\s*[:：]\s*([\d,]+\.?\d*)", text)
    if m:
        return float(m.group(1).replace(",", ""))
    nums = re.findall(r"[\d,]+\.?\d*", text)
    if nums:
        try:
            return float(nums[-1].replace(",", ""))
        except ValueError:
            return None
    return None


def self_consistency(question, k=5, temperature=0.8):
    """여러 번 풀어 다수결로 답을 정한다 (이론편 23.6절)"""
    msgs = [{"role": "user",
             "content": f"{question}\n\n단계적으로 생각한 뒤 마지막에 '답: 숫자' 형식으로 답하세요."}]

    answers = ask_llm(msgs, max_tokens=300, temperature=temperature, n=k)
    if answers is None:
        return None, []

    if isinstance(answers, str):
        answers = [answers]

    numbers = [extract_number(a) for a in answers]
    valid = [n for n in numbers if n is not None]

    if not valid:
        return None, numbers

    counts = Counter(valid)
    winner, votes = counts.most_common(1)[0]
    return winner, valid


problem = problems[0]
print(f"문제: {problem['question']}")
print(f"정답: {problem['answer']}")
print()

result, all_answers = self_consistency(problem["question"], k=5)

if result is not None:
    print(f"5장의 풀이 결과: {all_answers}")
    print(f"다수결 답      : {result}")
    print(f"정답 여부      : {'맞음' if abs(result - problem['answer']) < 0.01 else '틀림'}")
else:
    print("(API 키 없음 — 위 5절의 수치 실험으로 원리를 확인했습니다)")
    print()
    print("구현 요점")
    print("  1) temperature 를 0보다 크게 (다양한 경로가 나오도록)")
    print("  2) n=k 로 여러 응답을 한 번에 받는다")
    print("  3) 각 응답에서 최종 답을 추출한다")
    print("  4) 가장 많이 나온 답을 고른다")

### `temperature`가 0이면 안 되는 이유

24장에서 다뤘듯 `temperature=0`은 항상 같은 답을 낸다.
**5번을 돌려도 똑같은 결과 5개**가 나오므로 다수결이 무의미하다.

Self-Consistency는 **다양한 풀이 경로를 얻는 것**이 전제다.
그래서 `temperature=0.7~1.0` 정도를 쓴다.

---

## 6. 추론 특화 모델 — 이론편 23.6절

최근에는 **추론 과정을 더 길게, 스스로 검토하며** 진행하도록 학습된 모델들이 나왔다.

| 구분 | 일반 모델 | 추론 특화 모델 |
|---|---|---|
| 응답 방식 | 바로 답변 | 내부적으로 길게 생각한 뒤 답변 |
| 출력 토큰 | 수십~수백 | **수천 이상** |
| 속도 | 빠름 | 느림 |
| 비용 | 낮음 | 높음 |
| 잘하는 것 | 일반 대화, 요약 | 수학, 논리, 코딩 |

**3절의 "계산 자원" 관점**으로 보면 이해가 쉽다. 생각하는 토큰을 많이 쓰면
그만큼 신경망을 여러 번 통과하므로 어려운 문제를 풀 여지가 생긴다.

In [ ]:
import numpy as np

print("=" * 70)
print("추론 토큰과 비용")
print("=" * 70)
print()
print("추론 특화 모델은 답변 전에 '생각'하는 토큰을 많이 쓴다.")
print("그 토큰도 과금 대상이다 (25장 6절).")
print()

# 가정한 단가로 비교 (실제 값이 아님)
IN_PRICE, OUT_PRICE = 0.15, 0.60      # 100만 토큰당 달러 (가정)

print(f"{'방식':<24}{'입력':<10}{'출력':<12}{'비용(가정)':<14}{'특징'}")
print("-" * 70)
cases = [
    ("바로 답변",        100, 20,   "빠름"),
    ("CoT",             100, 200,  "중간"),
    ("Self-Consistency(5)", 500, 1000, "5배 호출"),
    ("추론 특화 모델",     100, 3000, "내부 사고 포함"),
]
for name, pt, ct, note in cases:
    cost = pt/1e6*IN_PRICE + ct/1e6*OUT_PRICE
    print(f"{name:<24}{pt:<10}{ct:<12}${cost:<13.6f}{note}")
print("-" * 70)
print()
print("정확도를 높이려면 비용과 시간을 내주어야 한다.")
print()
print("실무에서의 선택")
print("  간단한 작업 → 일반 모델, CoT 없이")
print("  중간 난이도 → CoT")
print("  중요하고 어려움 → Self-Consistency 또는 추론 모델")

---

## 7. 한계와 주의점 — 이론편 23.6절

CoT가 만능은 아니다. 이론편 23.6절에서 짚은 한계들을 정리한다.

In [ ]:
print("=" * 78)
print("CoT의 한계 (이론편 23.6절)")
print("=" * 78)
print()
print(f"{'한계':<24}{'설명':<32}{'대응'}")
print("-" * 78)
limits = [
    ("그럴듯한 오답",     "논리가 매끄러운데 결론이 틀림",  "답을 별도로 검증"),
    ("사후 정당화",       "먼저 답을 정하고 이유를 붙임",   "Self-Consistency"),
    ("쉬운 문제에 역효과", "단순 질문에 불필요한 장황함",   "문제 난이도로 분기"),
    ("비용·속도",         "토큰이 크게 늘어남",           "필요할 때만 사용"),
    ("작은 모델에서 미미", "일정 규모 이상에서 효과",       "모델 크기 고려"),
]
for a, b, c in limits:
    print(f"{a:<24}{b:<32}{c}")
print("-" * 78)
print()
print("[가장 주의할 것] '그럴듯한 오답'")
print()
print("  CoT는 풀이 과정을 보여주므로 신뢰감을 준다.")
print("  하지만 과정이 매끄럽다고 답이 맞는 것은 아니다.")
print("  중간 계산 하나가 틀리면 이후가 전부 어긋난다.")
print()
print("  → 3절의 p^n 이 그것을 말한다. 단계가 늘면 어딘가 틀릴 확률도 는다.")

In [ ]:
print("=" * 70)
print("답을 검증하는 방법")
print("=" * 70)
print()
print("[1] 계산은 코드로 넘긴다")
print("  모델에게 계산식을 만들게 하고, 실제 계산은 Python 이 한다.")
print("  → 37장의 도구 호출(Function Calling)")
print()

# 실제로 해 보기
print("예시: 모델이 식을 만들고 코드가 계산")
expression = "12 * 7 / 5"          # 모델이 만들었다고 가정
result = eval(expression)
print(f"  모델이 만든 식: {expression}")
print(f"  Python 계산   : {result}")
print(f"  정답          : {problems[0]['answer']}")
print(f"  일치          : {abs(result - problems[0]['answer']) < 0.01}")
print()
print("  이 방식이면 계산 실수가 원천적으로 사라진다.")
print()

print("[2] 역방향 검증")
print("  구한 답을 문제에 대입해 맞는지 확인하게 한다.")
print()
print("[3] 여러 방법으로 풀기")
print("  다른 접근으로 풀어 같은 답이 나오는지 본다.")
print()
print("-" * 70)
print("공통점: **모델의 말을 그대로 믿지 않는다**")
print("  23번 RAG의 출처 표시와 같은 정신이다.")

---

## 8. 정리

### 확인한 내용

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 23.6 | 단계 수와 정답률 ($p^n$) | 계산 ✓ |
| 23.6 | CoT가 각 단계를 쉽게 만듦 | 그래프 확인 ✓ |
| **23.6** | **Self-Consistency 효과** | **이항분포로 계산** ✓ |
| 23.6 | p<0.5면 다수결이 역효과 | 확인 ✓ |
| 23.6 | CoT의 한계 | 정리 ✓ |

### CoT가 작동하는 세 이유

| 이유 | 설명 |
|---|---|
| 계산 자원 | 토큰마다 신경망 1회 — 길게 쓰면 많이 계산 |
| 중간 결과 보존 | 문맥에 적어 두면 다시 계산할 필요 없음 |
| **단계 쉬워짐** | **각 단계의 p가 올라 전체 정답률 상승** |

### 기억할 것

| 항목 | 요점 |
|---|---|
| Zero-shot CoT | "단계적으로 생각하자" 한 문장 |
| Few-shot CoT | 풀이 예시로 형식까지 제어 |
| Self-Consistency | **p > 0.5 여야 효과** |
| temperature | 0이면 다수결 무의미 — 0.7~1.0 |
| 수확 체감 | 5~10회가 실용적 |
| 그럴듯한 오답 | 과정이 매끄러워도 답은 틀릴 수 있음 |
| 검증 | 계산은 코드로 넘기기 (37장) |

### 다음 장

**36. Multimodal AI — 이미지와 텍스트를 함께** — 이론편 31장.
텍스트만 다루던 것에서 벗어나 **이미지와 텍스트를 함께** 다룬다.
27장의 임베딩 개념이 이미지로 확장된다.